In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path
import flammkuchen as fl

In [ ]:
class GroupDecoder:
    def __init__(self, hidden_layer_sizes=(100, 50), random_state=42):
        self.scaler = StandardScaler()
        self.classifier = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            random_state=random_state,
            max_iter=1000
        )
        
    def prepare_data(self, neural_data, position_matrix, train_fraction=0.8):
        """
        Prepare data for single group, preserving temporal structure.
        
        Parameters:
        neural_data: array of shape (n_timepoints, n_neurons)
        position_matrix: array of shape (n_positions, n_timepoints)
        train_fraction: fraction of data to use for training
        """
        # Get stimulus times and positions
        stim_mask = np.any(position_matrix, axis=0)
        positions = np.argmax(position_matrix, axis=0)[stim_mask]
        neural_responses = neural_data[stim_mask]
        
        # Find split point preserving temporal order
        n_samples = len(positions)
        split_idx = int(n_samples * train_fraction)
        
        # Split the data temporally
        X_train = neural_responses[:split_idx]
        X_test = neural_responses[split_idx:]
        y_train = positions[:split_idx]
        y_test = positions[split_idx:]
        
        # Scale the data
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        
        return X_train, X_test, y_train, y_test
    
    def train(self, X_train, y_train):
        """Train the decoder."""
        self.classifier.fit(X_train, y_train)
    
    def predict(self, X):
        """Make predictions on new data."""
        X_scaled = self.scaler.transform(X)
        return self.classifier.predict(X_scaled)
    
    def predict_proba(self, X):
        """Get prediction probabilities."""
        X_scaled = self.scaler.transform(X)
        return self.classifier.predict_proba(X_scaled)
    
    def plot_temporal_predictions(self, X_test, y_test, window_size=100):
        """Plot temporal sequence of predictions vs true positions."""
        y_pred = self.predict(X_test)
        
        plt.figure(figsize=(15, 5))
        plt.plot(y_test, 'b-', label='True Position', alpha=0.5)
        plt.plot(y_pred, 'r.', label='Predicted Position', markersize=10)
        plt.title('Temporal Decoder Performance')
        plt.xlabel('Time (samples)')
        plt.ylabel('Position')
        plt.legend()
        plt.grid(True)
        plt.ylim(-0.5, 7.5)  # Assuming 8 positions
        plt.show()

class MultiGroupDecoder:
    def __init__(self, n_groups):
        self.decoders = [GroupDecoder() for _ in range(n_groups)]
        
    def train_all(self, neural_data_list, position_matrix_list, train_fraction=0.8):
        """
        Train all group decoders and evaluate their performance.
        
        Returns:
        group_results: list of dictionaries containing results for each group
        """
        group_results = []
        
        for i, (decoder, neural_data, position_matrix) in enumerate(zip(
            self.decoders, neural_data_list, position_matrix_list)):
            
            print(f"\nProcessing Group {i+1}/{len(neural_data_list)}")
            
            # Prepare data for this group
            X_train, X_test, y_train, y_test = decoder.prepare_data(
                neural_data, 
                position_matrix,
                train_fraction
            )
            
            # Train decoder
            print(f"Training decoder for group {i+1}...")
            decoder.train(X_train, y_train)
            
            # Evaluate
            y_pred = decoder.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            conf_mat = confusion_matrix(y_test, y_pred)
            
            # Store results
            group_results.append({
                'group_id': i,
                'accuracy': accuracy,
                'confusion_matrix': conf_mat,
                'X_test': X_test,
                'y_test': y_test,
                'y_pred': y_pred,
                'n_neurons': neural_data.shape[1]
            })
            
            print(f"Group {i+1} accuracy: {accuracy:.3f}")
        
        return group_results
    
    def plot_group_results(self, group_results):
        """Plot results for all groups."""
        n_groups = len(group_results)
        
        # Plot accuracies
        plt.figure(figsize=(10, 5))
        accuracies = [r['accuracy'] for r in group_results]
        neurons = [r['n_neurons'] for r in group_results]
        plt.bar(range(n_groups), accuracies)
        plt.xlabel('Group')
        plt.ylabel('Accuracy')
        plt.title('Decoder Accuracy by Group')
        
        # Add number of neurons as text
        for i, (acc, n) in enumerate(zip(accuracies, neurons)):
            plt.text(i, acc, f'n={n}', ha='center', va='bottom')
        
        plt.show()
        
        # Plot confusion matrices
        n_cols = min(3, n_groups)
        n_rows = (n_groups + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
        if n_groups == 1:
            axes = np.array([axes])
        axes = axes.flatten()
        
        for i, results in enumerate(group_results):
            if i < len(axes):
                sns.heatmap(results['confusion_matrix'], 
                           annot=True, fmt='d', cmap='Blues', ax=axes[i])
                axes[i].set_title(f'Group {i} Confusion Matrix\n'
                                f'Accuracy: {results["accuracy"]:.3f}')
                
                # Make empty subplots invisible
                if i >= len(group_results):
                    axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.show()
        
        # Plot temporal predictions for each group
        for i, (decoder, results) in enumerate(zip(self.decoders, group_results)):
            print(f"\nTemporal predictions for Group {i}:")
            decoder.plot_temporal_predictions(results['X_test'], results['y_test'])

def demo_multi_group_decoder(neural_data_list, position_matrix_list, train_fraction=0.8):
    """
    Demonstrate the multi-group decoder.
    
    Parameters:
    neural_data_list: list of arrays, each of shape (n_timepoints, n_neurons)
    position_matrix_list: list of arrays, each of shape (n_positions, n_timepoints)
    train_fraction: fraction of data to use for training
    """
    # Initialize multi-group decoder
    n_groups = len(neural_data_list)
    decoder = MultiGroupDecoder(n_groups)
    
    # Train and evaluate
    print("Training decoders for all groups...")
    group_results = decoder.train_all(
        neural_data_list,
        position_matrix_list,
        train_fraction
    )
    
    # Plot results
    decoder.plot_group_results(group_results)
    
    # Print summary
    print("\nSummary of results:")
    print("-------------------")
    accuracies = [r['accuracy'] for r in group_results]
    neurons = [r['n_neurons'] for r in group_results]
    
    for i, (acc, n) in enumerate(zip(accuracies, neurons)):
        print(f"Group {i}: Accuracy = {acc:.3f} ({n} neurons)")
    
    print(f"\nMean accuracy across groups: {np.mean(accuracies):.3f} ± {np.std(accuracies):.3f}")
    print(f"Combined number of neurons: {sum(neurons)}")
    
    return decoder, group_results


In [ ]:
master =  Path(r"Z:\Hagar and Ot\e0075\habenula")
fish_list = list(master.glob("*_f*"))

In [ ]:
fish = fish_list[-2] / "suite2p"
planes = list(fish.glob("*00*"))
    
neural_data_groups_l = []
neural_data_groups_r = []
stimulus_data_groups = []

for plane in planes:
    
    traces = fl.load(plane / 'filtered_traces.h5')['undetr']
    regs = fl.load(plane / 'sensory_regressors_cells.h5')['regressors']

    hab_coords_l = fl.load(plane / 'habenula_coords.h5')['lhab_coords']
    hab_coords_r = fl.load(plane / 'habenula_coords.h5')['rhab_coords']

    habenula_traces_l = traces[:,hab_coords_l]
    habenula_traces_r = traces[:,hab_coords_r]
    
    neural_data_groups_l =  neural_data_groups_l + [habenula_traces_l]
    neural_data_groups_r =  neural_data_groups_r + [habenula_traces_r]
    
    stimulus_data_groups = stimulus_data_groups + [regs]

In [ ]:
decoder, results = demo_multi_group_decoder(
    neural_data_groups_r,
    stimulus_data_groups,
    train_fraction=0.8  
)

In [ ]:
class PopulationDecoder:
    def __init__(self, hidden_layer_sizes=(100, 50), random_state=42):
        self.scaler = StandardScaler()
        self.classifier = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            random_state=random_state,
            max_iter=1000
        )
    
    def prepare_population_data(self, neural_data_list, position_matrix_list):
        """
        Create dataset where each sample corresponds to one stimulus presentation,
        with zero-padding for neurons not recorded in that group.
        
        Parameters:
        neural_data_list: list of arrays, each of shape (n_timepoints, n_neurons_in_group)
        position_matrix_list: list of arrays, each of shape (n_positions, n_timepoints)
        """
        # Calculate total number of neurons
        total_neurons = sum(data.shape[1] for data in neural_data_list)
        
        all_responses = []
        all_positions = []
        
        # Keep track of which neurons belong to which group
        neuron_start_idx = 0
        
        # Process each group
        for neural_data, position_matrix in zip(neural_data_list, position_matrix_list):
            # Find stimulus presentations
            stim_mask = np.any(position_matrix, axis=0)
            positions = np.argmax(position_matrix, axis=0)[stim_mask]
            group_responses = neural_data[stim_mask]
            
            # Create zero-padded responses for this group
            n_samples = len(positions)
            n_neurons_in_group = neural_data.shape[1]
            
            # Create zero array for all neurons
            padded_responses = np.zeros((n_samples, total_neurons))
            
            # Fill in the responses for this group's neurons
            padded_responses[:, neuron_start_idx:neuron_start_idx + n_neurons_in_group] = group_responses
            
            # Add to combined dataset
            all_responses.append(padded_responses)
            all_positions.append(positions)
            
            # Update starting index for next group
            neuron_start_idx += n_neurons_in_group
        
        # Combine all data
        X = np.vstack(all_responses)
        y = np.concatenate(all_positions)
        
        return X, y
    
    def train_and_evaluate(self, X, y, train_fraction=0.8):
        """
        Train and evaluate the decoder on the combined data.
        Uses temporal split to maintain trial structure.
        """
        # Split data temporally
        n_samples = len(y)
        split_idx = int(n_samples * train_fraction)
        
        X_train = X[:split_idx]
        X_test = X[split_idx:]
        y_train = y[:split_idx]
        y_test = y[split_idx:]
        
        # Scale and train
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        self.classifier.fit(X_train_scaled, y_train)
        y_pred = self.classifier.predict(X_test_scaled)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        conf_mat = confusion_matrix(y_test, y_pred)
        
        return {
            'accuracy': accuracy,
            'confusion_matrix': conf_mat,
            'y_test': y_test,
            'y_pred': y_pred,
            'X_test': X_test_scaled
        }

    def plot_results(self, results):
        """Plot decoder performance."""
        # 1. Confusion Matrix
        plt.figure(figsize=(10, 8))
        sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', cmap='Blues')
        plt.title(f'Population Decoder Confusion Matrix\nAccuracy: {results["accuracy"]:.3f}')
        plt.xlabel('Predicted Position')
        plt.ylabel('True Position')
        plt.show()
        
        # 2. Temporal Predictions
        plt.figure(figsize=(15, 5))
        plt.plot(results['y_test'], 'b-', label='True Position', alpha=0.5)
        plt.plot(results['y_pred'], 'r.', label='Predicted Position', markersize=10)
        plt.title('Population Decoder Performance Over Time')
        plt.xlabel('Test Sample')
        plt.ylabel('Position')
        plt.legend()
        plt.grid(True)
        plt.ylim(-0.5, 7.5)  # Assuming 8 positions
        plt.show()
        
        # 3. Position-specific accuracy
        conf_mat = results['confusion_matrix']
        pos_accuracy = conf_mat.diagonal() / conf_mat.sum(axis=1)
        
        plt.figure(figsize=(10, 5))
        plt.plot(pos_accuracy, 'o-')
        plt.xlabel('Position')
        plt.ylabel('Accuracy')
        plt.title('Decoding Accuracy by Position')
        plt.grid(True)
        plt.show()

def demo_population_decoder(neural_data_list, position_matrix_list, train_fraction=0.8):
    """Run population decoder analysis."""
    # Initialize decoder
    decoder = PopulationDecoder()
    
    # Print initial group sizes
    print("Group sizes:")
    for i, data in enumerate(neural_data_list):
        print(f"Group {i}: {data.shape[1]} neurons, {data.shape[0]} timepoints")
    
    # Prepare combined data
    print("\nPreparing population data...")
    X, y = decoder.prepare_population_data(neural_data_list, position_matrix_list)
    print(f"Combined dataset: {X.shape[0]} samples, {X.shape[1]} neurons")
    
    # Train and evaluate
    print("\nTraining and evaluating population decoder...")
    results = decoder.train_and_evaluate(X, y, train_fraction)
    
    # Plot results
    decoder.plot_results(results)
    
    # Print summary
    print("\nPopulation Decoder Summary:")
    print(f"Total number of neurons: {X.shape[1]}")
    print(f"Total number of samples: {X.shape[0]}")
    print(f"Overall accuracy: {results['accuracy']:.3f}")
    
    return decoder, results


In [ ]:
decoder, results = demo_population_decoder(neural_data_groups_r, stimulus_data_groups)